# Objective Thresholds For Pareto Mode

In Ax multi-objective optimization, objective thresholds define the region of
interest. They act like a reference point:

- below the threshold: not interesting enough
- above the threshold: contributes to the preferred Pareto region

In the service API, thresholds are attached through `ObjectiveProperties`.


In [1]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties

warnings.filterwarnings("ignore", message=".*RandomModelBridge does not support prediction.*")

QUALITY_THRESHOLD = 0.70
STABILITY_THRESHOLD = 0.70

def evaluate_material(x: float, y: float) -> dict[str, tuple[float, float]]:
    quality = 1.0 - ((x - 0.20) ** 2 + (y - 0.80) ** 2)
    stability = 1.0 - ((x - 0.80) ** 2 + (y - 0.20) ** 2)
    return {
        "quality": (quality, 0.0),
        "stability": (stability, 0.0),
    }

client = AxClient(
    random_seed=0,
    enforce_sequential_optimization=False,
    verbose_logging=False,
)
client.create_experiment(
    parameters=[
        {"name": "x", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
        {"name": "y", "type": "range", "bounds": [0.0, 1.0], "value_type": "float"},
    ],
    objectives={
        "quality": ObjectiveProperties(minimize=False, threshold=QUALITY_THRESHOLD),
        "stability": ObjectiveProperties(minimize=False, threshold=STABILITY_THRESHOLD),
    },
)

for _ in range(12):
    params, trial_index = client.get_next_trial()
    client.complete_trial(trial_index, evaluate_material(**params))

df = client.get_trials_data_frame()[["trial_index", "x", "y", "quality", "stability"]].copy()
df["inside_threshold_box"] = (
    (df["quality"] >= QUALITY_THRESHOLD) &
    (df["stability"] >= STABILITY_THRESHOLD)
)

threshold_summary = [
    (ot.metric.name, ot.bound)
    for ot in client.experiment.optimization_config.objective_thresholds
]
pareto = client.get_pareto_optimal_parameters()

print("Stored objective thresholds:", threshold_summary)
print("Number of observed Pareto-optimal trials:", len(pareto))

pareto_rows = []
for trial_index, (params, values) in pareto.items():
    means = values[0]
    pareto_rows.append(
        {
            "trial_index": trial_index,
            "x": params["x"],
            "y": params["y"],
            "quality": float(means["quality"]),
            "stability": float(means["stability"]),
        }
    )

pareto_df = pd.DataFrame(pareto_rows).sort_values(["quality", "stability"], ascending=False)
pareto_df


[WARNING 03-17 12:05:05] ax.service.utils.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.
[WARNING 03-17 12:05:05] ax.service.ax_client: Random seed set to 0. Note that this setting only affects the Sobol quasi-random generator and BoTorch-powered Bayesian optimization models. For the latter models, setting random seed to the same number for two optimizations will make the generated trials similar, but not exactly the same, and over time the trials will diverge more.
[INFO 03-17 12:05:05] ax.service.utils.instantiation: Created search space: SearchSpace(parameters=[RangeParameter(name='x', parameter_type=FLOAT, range=[0.0, 1.0]), RangeParameter(name='y', parameter_type=FLOAT, range=[0.0, 1.0])], parameter_constraints=[]).
[INFO 03-17 12:05:05] ax.modelbridge.dispatch_u

Stored objective thresholds: [('quality', 0.7), ('stability', 0.7)]
Number of observed Pareto-optimal trials: 7


,trial_index,x,y,quality,stability
5,0,0.475107,0.592524,0.881269,0.740344
3,8,0.486883,0.550293,0.855368,0.779264
2,11,0.492814,0.529375,0.840996,0.797133
0,6,0.502654,0.512383,0.825712,0.814039
1,10,0.515118,0.483471,0.800445,0.838490
4,7,0.525504,0.452109,0.773045,0.861094
6,9,0.549579,0.422968,0.735649,0.887558


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

colors = df["inside_threshold_box"].map({True: "tab:blue", False: "lightgray"})
ax.scatter(df["quality"], df["stability"], c=colors, s=90)

if not pareto_df.empty:
    ax.scatter(
        pareto_df["quality"],
        pareto_df["stability"],
        facecolors="none",
        edgecolors="black",
        s=180,
        linewidths=1.5,
        label="observed Pareto set",
    )

ax.axvline(QUALITY_THRESHOLD, linestyle="--", color="tab:red", label="quality threshold")
ax.axhline(STABILITY_THRESHOLD, linestyle="--", color="tab:green", label="stability threshold")
ax.set_xlabel("quality")
ax.set_ylabel("stability")
ax.set_title("Multi-objective observations with threshold lines")
ax.legend(loc="lower left")
plt.show()


## Interpretation

The dashed lines are not hard feasibility constraints in the same way outcome constraints are.
They define the part of objective space that Ax should treat as worthwhile when optimizing
the Pareto frontier and hypervolume.
